# Notebook 01: Load Data & Smart Merging

This notebook loads LogFile and UsnJrnl CSVs, applies Oh et al. 2024 filtering methodology, and performs smart union merging.

**Optional Validation Mode**: If a Suspicious CSV path is provided, it will merge with suspicious records to extract ground truth labels and zero nanoseconds patterns for validation.

## Steps:
1. Load raw LogFile and UsnJrnl CSVs
2. Filter LogFile (Time Reversal + Update events)
3. Filter UsnJrnl (Basic_Info_Change events)
4. (Optional) Load and merge with Suspicious CSV
5. Smart union merge (outer join on filename)
6. Aggregate to file level
7. Extract zero nanoseconds from suspicious detail
8. Save merged dataset

## Output:
- `data_merged.csv` - Merged and filtered dataset ready for feature engineering


In [370]:
# Cell 2: User Configuration

# INPUT PATHS - Edit these to point to your dataset
LOGFILE_CSV_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/added datasets/testing/logfile/14-Winnti43b-LogFile.csv'
USNJRNL_CSV_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/added datasets/testing/usnjrnl/14-Winnti43b-UsnJrnl.csv'

# OPTIONAL: Suspicious CSV for validation (set to None if not available)
SUSPICIOUS_CSV_PATH = None
# OUTPUT PATH
OUTPUT_DIR = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/14-W43b'
OUTPUT_MERGED_CSV = f'{OUTPUT_DIR}/data_merged.csv'

print("Configuration loaded successfully")
print(f"LogFile CSV: {LOGFILE_CSV_PATH}")
print(f"UsnJrnl CSV: {USNJRNL_CSV_PATH}")
print(f"Suspicious CSV: {SUSPICIOUS_CSV_PATH if SUSPICIOUS_CSV_PATH else 'NOT PROVIDED (production mode)'}")
print(f"Output directory: {OUTPUT_DIR}")


Configuration loaded successfully
LogFile CSV: /Users/soni/Github/Digital-Detectives_Thesis/data/added datasets/testing/logfile/14-Winnti43b-LogFile.csv
UsnJrnl CSV: /Users/soni/Github/Digital-Detectives_Thesis/data/added datasets/testing/usnjrnl/14-Winnti43b-UsnJrnl.csv
Suspicious CSV: NOT PROVIDED (production mode)
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/14-W43b


In [371]:
# Cell 3: Import Libraries and Column Mappings

import pandas as pd
import numpy as np
import os
from pathlib import Path

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LogFile column mapping (standardize naming)
LOGFILE_COLUMNS = {
    'LSN': 'lf_lsn',
    'Event Time(UTC+8)': 'lf_event_time',
    'Event': 'lf_event',
    'Detail': 'lf_detail',
    'Filename': 'lf_filename',
    'Full Path': 'lf_full_path',
    'CreationTime': 'lf_creation_time',
    'ModifiedTime': 'lf_modified_time',
    'MFTModifiedTime': 'lf_mft_modified_time',
    'AccessedTime': 'lf_accessed_time'
}

# UsnJrnl column mapping (standardize naming)
USNJRNL_COLUMNS = {
    'USN': 'usn_usn',
    'TimeStamp(UTC+8)': 'usn_event_time',
    'EventInfo': 'usn_event_info',
    'File/Directory Name': 'usn_filename',
    'FullPath': 'usn_full_path',
    'FileReferenceNumber': 'usn_file_reference_number',
    'ParentFileReferenceNumber': 'usn_parent_file_reference_number'
}

# Suspicious CSV column mapping
SUSPICIOUS_COLUMNS = {
    'source': 'suspicious_source',
    'lsn/usn': 'suspicious_lsn_usn',
    'category': 'suspicious_category',
    'detail': 'suspicious_detail'
}

print("Libraries imported and column mappings defined")


Libraries imported and column mappings defined


In [372]:
# Cell 4: Helper Functions

def filter_logfile_timestamp_changes(lf_df):
    """
    Filter LogFile to Time Reversal + Update events only.
    Expected reduction: ~96.8% (e.g., 39,077 -> 1,235 for 01-PE)
    """
    # Pattern 1: Time Reversal events
    time_reversal = lf_df[
        lf_df['lf_event'].str.contains('Time Reversal', na=False, case=False)
    ].copy()
    
    # Pattern 2: Update events (Update Resident Value, Updating MFTModified Time, etc.)
    update_events = lf_df[
        lf_df['lf_event'].str.contains('Update', na=False, case=False)
    ].copy()
    
    # Combine and remove duplicates
    filtered = pd.concat([time_reversal, update_events]).drop_duplicates(subset=['lf_lsn'])
    
    reduction_pct = (1 - len(filtered) / len(lf_df)) * 100 if len(lf_df) > 0 else 0
    print(f"LogFile filtering:")
    print(f"  Raw records: {len(lf_df):,}")
    print(f"  Time Reversal events: {len(time_reversal):,}")
    print(f"  Update events: {len(update_events):,}")
    print(f"  Filtered records: {len(filtered):,}")
    print(f"  Reduction: {reduction_pct:.1f}%")
    
    return filtered

def find_basic_detection_pattern(usn_df):
    """
    Filter UsnJrnl to Basic_Info_Change events only.
    Expected reduction: ~92.4% (e.g., 316,817 -> 24,002 for 01-PE)
    """
    filtered = usn_df[
        usn_df['usn_event_info'].str.contains('Basic_Info_Change', na=False, case=False)
    ].copy()
    
    reduction_pct = (1 - len(filtered) / len(usn_df)) * 100 if len(usn_df) > 0 else 0
    print(f"\nUsnJrnl filtering:")
    print(f"  Raw records: {len(usn_df):,}")
    print(f"  Basic_Info_Change events: {len(filtered):,}")
    print(f"  Reduction: {reduction_pct:.1f}%")
    
    return filtered

print("Helper functions defined")


Helper functions defined


In [373]:
# Cell 5: Load Raw CSVs

print("Loading raw CSVs...")

# Load LogFile
lf_raw = pd.read_csv(LOGFILE_CSV_PATH, low_memory=False)
print(f"LogFile loaded: {len(lf_raw):,} records")

# Load UsnJrnl
usn_raw = pd.read_csv(USNJRNL_CSV_PATH, low_memory=False)
print(f"UsnJrnl loaded: {len(usn_raw):,} records")

# Standardize column names (only rename columns that exist)
lf_raw = lf_raw.rename(columns={k: v for k, v in LOGFILE_COLUMNS.items() if k in lf_raw.columns})
usn_raw = usn_raw.rename(columns={k: v for k, v in USNJRNL_COLUMNS.items() if k in usn_raw.columns})

print("\nColumn names standardized")
print(f"LogFile columns: {list(lf_raw.columns)[:5]}...")
print(f"UsnJrnl columns: {list(usn_raw.columns)[:5]}...")


Loading raw CSVs...
LogFile loaded: 27,741 records
UsnJrnl loaded: 248,399 records

Column names standardized
LogFile columns: ['lf_lsn', 'EventTime(UTC+8)', 'lf_event', 'lf_detail', 'File/Directory Name']...
UsnJrnl columns: ['usn_event_time', 'usn_usn', 'usn_filename', 'usn_full_path', 'usn_event_info']...


In [374]:
# Cell 6: Filter LogFile for Timestamp Changes

lf_filtered = filter_logfile_timestamp_changes(lf_raw)


LogFile filtering:
  Raw records: 27,741
  Time Reversal events: 1,081
  Update events: 1
  Filtered records: 1,082
  Reduction: 96.1%


In [375]:
# Cell 7: Filter UsnJrnl for Basic Info Changes

usn_filtered = find_basic_detection_pattern(usn_raw)



UsnJrnl filtering:
  Raw records: 248,399
  Basic_Info_Change events: 15,900
  Reduction: 93.6%


In [376]:
# Cell 8: Load Suspicious CSV (Optional - For Validation)

suspicious_df = None

if SUSPICIOUS_CSV_PATH and os.path.exists(SUSPICIOUS_CSV_PATH):
    print("\n" + "="*80)
    print("VALIDATION MODE: Loading Suspicious CSV for ground truth labels")
    print("="*80)
    
    suspicious_df = pd.read_csv(SUSPICIOUS_CSV_PATH)
    print(f"Suspicious CSV loaded: {len(suspicious_df):,} records")
    
    # Standardize column names
    suspicious_df = suspicious_df.rename(columns={k: v for k, v in SUSPICIOUS_COLUMNS.items() if k in suspicious_df.columns})
    
    # Convert lsn/usn to numeric
    suspicious_df['suspicious_lsn_usn'] = pd.to_numeric(suspicious_df['suspicious_lsn_usn'], errors='coerce')
    
    # Split into LogFile and UsnJrnl suspicious records
    suspicious_lf = suspicious_df[suspicious_df['suspicious_source'] == 'logfile'].copy()
    suspicious_usn = suspicious_df[suspicious_df['suspicious_source'] == 'usnjrnl'].copy()
    
    print(f"  LogFile suspicious: {len(suspicious_lf):,} records")
    print(f"  UsnJrnl suspicious: {len(suspicious_usn):,} records")
    
    # Display sample suspicious records
    print("\nSample suspicious records:")
    print(suspicious_df[['suspicious_source', 'suspicious_lsn_usn', 'suspicious_category']].head(3))
    
else:
    print("\n" + "="*80)
    print("PRODUCTION MODE: No Suspicious CSV provided")
    print("="*80)
    print("Running in production mode (no ground truth labels)")
    suspicious_lf = None
    suspicious_usn = None



PRODUCTION MODE: No Suspicious CSV provided
Running in production mode (no ground truth labels)


In [377]:
# Cell 9: Merge LogFile with Suspicious (by LSN)

if suspicious_lf is not None and len(suspicious_lf) > 0:
    print("\nMerging LogFile with Suspicious CSV by LSN...")
    
    before_merge = len(lf_filtered)
    
    # Left join to preserve all filtered records
    lf_filtered = pd.merge(
        lf_filtered,
        suspicious_lf,
        left_on='lf_lsn',
        right_on='suspicious_lsn_usn',
        how='left',
        suffixes=('', '_suspicious')
    )
    
    # Count how many LogFile records have suspicious labels
    lf_suspicious_count = lf_filtered['suspicious_category'].notna().sum()
    
    print(f"  LogFile records before merge: {before_merge:,}")
    print(f"  LogFile records after merge: {len(lf_filtered):,}")
    print(f"  LogFile records with suspicious labels: {lf_suspicious_count:,}")
    
else:
    print("\nSkipping LogFile + Suspicious merge (no suspicious data)")
    # Add placeholder columns
    lf_filtered['suspicious_source'] = None
    lf_filtered['suspicious_category'] = None
    lf_filtered['suspicious_detail'] = None



Skipping LogFile + Suspicious merge (no suspicious data)


In [378]:
# Cell 10: Merge UsnJrnl with Suspicious (by USN)

if suspicious_usn is not None and len(suspicious_usn) > 0:
    print("\nMerging UsnJrnl with Suspicious CSV by USN...")
    
    before_merge = len(usn_filtered)
    
    # Left join to preserve all filtered records
    usn_filtered = pd.merge(
        usn_filtered,
        suspicious_usn,
        left_on='usn_usn',
        right_on='suspicious_lsn_usn',
        how='left',
        suffixes=('', '_suspicious')
    )
    
    # Count how many UsnJrnl records have suspicious labels
    usn_suspicious_count = usn_filtered['suspicious_category'].notna().sum()
    
    print(f"  UsnJrnl records before merge: {before_merge:,}")
    print(f"  UsnJrnl records after merge: {len(usn_filtered):,}")
    print(f"  UsnJrnl records with suspicious labels: {usn_suspicious_count:,}")
    
    # Display known timestomped files
    if usn_suspicious_count > 0:
        print("\nKnown timestomped files from Suspicious CSV:")
        timestomped = usn_filtered[usn_filtered['suspicious_category'].notna()][
            ['usn_usn', 'usn_filename', 'suspicious_category', 'suspicious_detail']
        ]
        print(timestomped.to_string(index=False, max_rows=15))
    
else:
    print("\nSkipping UsnJrnl + Suspicious merge (no suspicious data)")
    # Add placeholder columns
    usn_filtered['suspicious_source'] = None
    usn_filtered['suspicious_category'] = None
    usn_filtered['suspicious_detail'] = None



Skipping UsnJrnl + Suspicious merge (no suspicious data)


In [379]:
# Cell 10.5: Diagnostic - Check Available Columns

print("\n" + "="*80)
print("DIAGNOSTIC: Available Columns After Merge")
print("="*80)

print("\nLogFile columns:")
lf_cols = [col for col in lf_filtered.columns if col.startswith('lf_')]
print(f"  Total: {len(lf_cols)}")
print(f"  Columns: {lf_cols[:10]}...")

print("\nUsnJrnl columns:")
usn_cols = [col for col in usn_filtered.columns if col.startswith('usn_')]
print(f"  Total: {len(usn_cols)}")
print(f"  Columns: {usn_cols[:10]}...")

# Check for filename-like columns
print("\nSearching for filename-related columns:")
all_lf_cols = list(lf_filtered.columns)
all_usn_cols = list(usn_filtered.columns)

filename_candidates_lf = [col for col in all_lf_cols if 'file' in col.lower() or 'name' in col.lower()]
filename_candidates_usn = [col for col in all_usn_cols if 'file' in col.lower() or 'name' in col.lower()]

print(f"  LogFile candidates: {filename_candidates_lf}")
print(f"  UsnJrnl candidates: {filename_candidates_usn}")



DIAGNOSTIC: Available Columns After Merge

LogFile columns:
  Total: 8
  Columns: ['lf_lsn', 'lf_event', 'lf_detail', 'lf_full_path', 'lf_creation_time', 'lf_modified_time', 'lf_mft_modified_time', 'lf_accessed_time']...

UsnJrnl columns:
  Total: 7
  Columns: ['usn_event_time', 'usn_usn', 'usn_filename', 'usn_full_path', 'usn_event_info', 'usn_file_reference_number', 'usn_parent_file_reference_number']...

Searching for filename-related columns:
  LogFile candidates: ['File/Directory Name']
  UsnJrnl candidates: ['usn_filename', 'FileAttribute', 'usn_file_reference_number', 'usn_parent_file_reference_number']


In [380]:
# Cell 11: Smart Union Merge (LogFile + UsnJrnl)

print("\n" + "="*80)
print("SMART UNION MERGE: Outer join on filename")
print("="*80)

# Create merge keys - handle missing filename columns gracefully
# LogFile: Try lf_filename first, fall back to extracting from full_path
if 'lf_filename' in lf_filtered.columns:
    lf_filtered['merge_key'] = lf_filtered['lf_filename'].fillna('').str.lower().str.strip()
elif 'lf_full_path' in lf_filtered.columns:
    # Extract filename from full path
    lf_filtered['merge_key'] = lf_filtered['lf_full_path'].fillna('').str.split('\\').str[-1].str.lower().str.strip()
else:
    # No filename available - use empty string (will be filtered out)
    lf_filtered['merge_key'] = ''
    print("  WARNING: No filename column found in LogFile!")

# UsnJrnl: Try usn_filename first, fall back to extracting from full_path
if 'usn_filename' in usn_filtered.columns:
    usn_filtered['merge_key'] = usn_filtered['usn_filename'].fillna('').str.lower().str.strip()
elif 'usn_full_path' in usn_filtered.columns:
    # Extract filename from full path
    usn_filtered['merge_key'] = usn_filtered['usn_full_path'].fillna('').str.split('\\').str[-1].str.lower().str.strip()
else:
    # No filename available - use empty string (will be filtered out)
    usn_filtered['merge_key'] = ''
    print("  WARNING: No filename column found in UsnJrnl!")

# Remove empty merge keys
lf_filtered = lf_filtered[lf_filtered['merge_key'] != ''].copy()
usn_filtered = usn_filtered[usn_filtered['merge_key'] != ''].copy()

print(f"LogFile records with valid filename: {len(lf_filtered):,}")
print(f"UsnJrnl records with valid filename: {len(usn_filtered):,}")

# Outer join to preserve records from BOTH sources
merged_df = pd.merge(
    lf_filtered,
    usn_filtered,
    on='merge_key',
    how='outer',
    suffixes=('_lf', '_usn')
)

print(f"\nMerged records (outer join): {len(merged_df):,}")

# Handle suspicious columns from both sources (prioritize non-null values)
if 'suspicious_category_lf' in merged_df.columns and 'suspicious_category_usn' in merged_df.columns:
    merged_df['suspicious_category'] = merged_df['suspicious_category_lf'].fillna(merged_df['suspicious_category_usn'])
    merged_df['suspicious_detail'] = merged_df['suspicious_detail_lf'].fillna(merged_df['suspicious_detail_usn'])
    merged_df['suspicious_source'] = merged_df['suspicious_source_lf'].fillna(merged_df['suspicious_source_usn'])
    
    # Drop duplicate suspicious columns
    merged_df = merged_df.drop(columns=[
        'suspicious_category_lf', 'suspicious_category_usn',
        'suspicious_detail_lf', 'suspicious_detail_usn',
        'suspicious_source_lf', 'suspicious_source_usn'
    ], errors='ignore')
elif 'suspicious_category' not in merged_df.columns:
    # No suspicious data at all - add placeholder columns
    merged_df['suspicious_category'] = None
    merged_df['suspicious_detail'] = None
    merged_df['suspicious_source'] = None



SMART UNION MERGE: Outer join on filename
LogFile records with valid filename: 1,064
UsnJrnl records with valid filename: 15,900

Merged records (outer join): 18,215


In [381]:
# Cell 12: Add Evidence Source Flags

# Count events per file (before aggregation)
merged_df['lf_event_count'] = merged_df.groupby('merge_key')['lf_lsn'].transform('count')
merged_df['usn_event_count'] = merged_df.groupby('merge_key')['usn_usn'].transform('count')

# Evidence flags
merged_df['has_logfile_evidence'] = merged_df['lf_event'].notna()
merged_df['has_usnjrnl_evidence'] = merged_df['usn_event_info'].notna()

# Cross-artifact validation score
def calculate_validation_score(row):
    score = 0
    if row['has_logfile_evidence']:
        score += 0.5
    if row['has_usnjrnl_evidence']:
        score += 0.5
    return score

merged_df['cross_artifact_validation_score'] = merged_df.apply(calculate_validation_score, axis=1)

print("\nEvidence source summary:")
print(f"  Records with LogFile evidence: {merged_df['has_logfile_evidence'].sum():,}")
print(f"  Records with UsnJrnl evidence: {merged_df['has_usnjrnl_evidence'].sum():,}")
print(f"  Records with BOTH (cross-artifact): {(merged_df['cross_artifact_validation_score'] == 1.0).sum():,}")



Evidence source summary:
  Records with LogFile evidence: 4,710
  Records with UsnJrnl evidence: 18,210
  Records with BOTH (cross-artifact): 4,705


In [382]:
# Cell 13: Aggregate to File Level (Keep First Event per File)

print("\n" + "="*80)
print("FILE-LEVEL AGGREGATION")
print("="*80)

print(f"Records before aggregation: {len(merged_df):,}")

# For files with suspicious labels, prioritize keeping the suspicious event
# Sort by suspicious_category (non-null first), then by LSN/USN
merged_df['has_suspicious_label'] = merged_df['suspicious_category'].notna()
merged_df = merged_df.sort_values(
    by=['merge_key', 'has_suspicious_label', 'lf_lsn', 'usn_usn'],
    ascending=[True, False, True, True],
    na_position='last'
)

# Keep first record per file (which will be suspicious event if it exists)
aggregated_df = merged_df.groupby('merge_key', as_index=False).first()

print(f"Unique files after aggregation: {len(aggregated_df):,}")

# Create final filename and full_path columns (handle missing columns)
if 'lf_filename' in aggregated_df.columns:
    aggregated_df['filename'] = aggregated_df['lf_filename'].fillna(aggregated_df.get('usn_filename', ''))
elif 'usn_filename' in aggregated_df.columns:
    aggregated_df['filename'] = aggregated_df['usn_filename']
else:
    # Extract from merge_key as fallback
    aggregated_df['filename'] = aggregated_df['merge_key']

if 'lf_full_path' in aggregated_df.columns:
    aggregated_df['full_path'] = aggregated_df['lf_full_path'].fillna(aggregated_df.get('usn_full_path', ''))
elif 'usn_full_path' in aggregated_df.columns:
    aggregated_df['full_path'] = aggregated_df['usn_full_path']
else:
    aggregated_df['full_path'] = ''

print(f"Files with filename: {aggregated_df['filename'].notna().sum():,}")
print(f"Files with full_path: {aggregated_df['full_path'].notna().sum():,}")



FILE-LEVEL AGGREGATION
Records before aggregation: 18,215
Unique files after aggregation: 3,535
Files with filename: 3,532
Files with full_path: 3,431


In [383]:
# Cell 14: Extract Zero Nanoseconds from Suspicious Detail

print("\n" + "="*80)
print("ZERO NANOSECONDS EXTRACTION")
print("="*80)

# Extract from LogFile Detail field (production-ready)
aggregated_df['zero_in_nanoseconds_lf'] = aggregated_df['lf_detail'].fillna('').str.contains(
    'Zero in 100-nanoseconds', case=False, na=False, regex=False
)

# Extract from Suspicious Detail field (validation only)
if 'suspicious_detail' in aggregated_df.columns:
    aggregated_df['zero_in_nanoseconds_suspicious'] = aggregated_df['suspicious_detail'].fillna('').str.contains(
        'Zero in.*100-nanoseconds', case=False, na=False, regex=True
    )
else:
    aggregated_df['zero_in_nanoseconds_suspicious'] = False

# Combined: True if EITHER source shows pattern
aggregated_df['zero_in_nanoseconds_combined'] = (
    aggregated_df['zero_in_nanoseconds_lf'] | 
    aggregated_df['zero_in_nanoseconds_suspicious']
)

print(f"Zero nanoseconds detected (LogFile Detail): {aggregated_df['zero_in_nanoseconds_lf'].sum():,}")
print(f"Zero nanoseconds detected (Suspicious Detail): {aggregated_df['zero_in_nanoseconds_suspicious'].sum():,}")
print(f"Zero nanoseconds detected (Combined): {aggregated_df['zero_in_nanoseconds_combined'].sum():,}")

if aggregated_df['zero_in_nanoseconds_combined'].sum() > 0:
    print("\nFiles with zero nanoseconds pattern:")
    zero_nano_files = aggregated_df[aggregated_df['zero_in_nanoseconds_combined']][
        ['filename', 'lf_lsn', 'usn_usn', 'zero_in_nanoseconds_lf', 'zero_in_nanoseconds_suspicious']
    ]
    print(zero_nano_files.head(10).to_string(index=False))



ZERO NANOSECONDS EXTRACTION
Zero nanoseconds detected (LogFile Detail): 747
Zero nanoseconds detected (Suspicious Detail): 0
Zero nanoseconds detected (Combined): 747

Files with zero nanoseconds pattern:
                                                        filename       lf_lsn     usn_usn  zero_in_nanoseconds_lf  zero_in_nanoseconds_suspicious
                                             164.4.7914.manifest 6292873925.0 998097856.0                    True                           False
66a1514ea0b833d9108f7ad1ec39a568cedcb46839f956ab330fb72791fd827d 6292061154.0 997937536.0                    True                           False
                                      _bowbow.cp38-win_amd64.pyd 6293002071.0 998109880.0                    True                           False
                                _cffi_backend.cp38-win_amd64.pyd 6293003388.0 998110360.0                    True                           False
                              _win32sysloader.cp38-win_amd64.pyd

In [384]:
# Cell 15: Create Ground Truth Labels

if 'suspicious_category' in aggregated_df.columns:
    # Flag files that appear in Suspicious CSV
    aggregated_df['is_flagged_suspicious'] = aggregated_df['suspicious_category'].notna()
    
    # Ground truth label: 1 if flagged, 0 otherwise
    aggregated_df['ground_truth_label'] = aggregated_df['is_flagged_suspicious'].astype(int)
    
    print("\n" + "="*80)
    print("GROUND TRUTH LABELS")
    print("="*80)
    print(f"Total files: {len(aggregated_df):,}")
    print(f"Flagged as suspicious: {aggregated_df['is_flagged_suspicious'].sum():,}")
    print(f"Ground truth positives: {aggregated_df['ground_truth_label'].sum():,}")
    
    # Breakdown by category
    if aggregated_df['is_flagged_suspicious'].sum() > 0:
        print("\nSuspicious categories breakdown:")
        print(aggregated_df[aggregated_df['is_flagged_suspicious']]['suspicious_category'].value_counts())
        
        print("\nKnown timestomped files in dataset:")
        known_files = aggregated_df[aggregated_df['ground_truth_label'] == 1][
            ['filename', 'usn_usn', 'zero_in_nanoseconds_combined', 'suspicious_category']
        ]
        print(known_files.to_string(index=False, max_rows=15))
else:
    print("\nNo ground truth labels (production mode)")
    aggregated_df['is_flagged_suspicious'] = False
    aggregated_df['ground_truth_label'] = 0



GROUND TRUTH LABELS
Total files: 3,535
Flagged as suspicious: 0
Ground truth positives: 0


In [385]:
# Cell 16: Save Merged Dataset

print("\n" + "="*80)
print("SAVING OUTPUT")
print("="*80)

# Save to CSV
aggregated_df.to_csv(OUTPUT_MERGED_CSV, index=False)

print(f"Saved merged dataset to: {OUTPUT_MERGED_CSV}")
print(f"Total records: {len(aggregated_df):,}")
print(f"Total columns: {len(aggregated_df.columns)}")
print(f"File size: {os.path.getsize(OUTPUT_MERGED_CSV) / 1024 / 1024:.2f} MB")



SAVING OUTPUT
Saved merged dataset to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/14-W43b/data_merged.csv
Total records: 3,535
Total columns: 40
File size: 3.37 MB


In [386]:
# Cell 17: Summary Statistics

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nData Reduction:")
print(f"  LogFile: {len(lf_raw):,} → {len(lf_filtered):,} ({(1 - len(lf_filtered)/len(lf_raw))*100:.1f}% reduction)")
print(f"  UsnJrnl: {len(usn_raw):,} → {len(usn_filtered):,} ({(1 - len(usn_filtered)/len(usn_raw))*100:.1f}% reduction)")
print(f"  Merged: {len(merged_df):,} records")
print(f"  Unique files: {len(aggregated_df):,} files")

print(f"\nEvidence Sources:")
print(f"  LogFile only: {((aggregated_df['has_logfile_evidence']) & (~aggregated_df['has_usnjrnl_evidence'])).sum():,}")
print(f"  UsnJrnl only: {((~aggregated_df['has_logfile_evidence']) & (aggregated_df['has_usnjrnl_evidence'])).sum():,}")
print(f"  Both sources (cross-artifact): {((aggregated_df['has_logfile_evidence']) & (aggregated_df['has_usnjrnl_evidence'])).sum():,}")

print(f"\nKey Features:")
print(f"  Zero nanoseconds: {aggregated_df['zero_in_nanoseconds_combined'].sum():,}")
print(f"  Cross-artifact validation score = 1.0: {(aggregated_df['cross_artifact_validation_score'] == 1.0).sum():,}")

if 'ground_truth_label' in aggregated_df.columns and aggregated_df['ground_truth_label'].sum() > 0:
    print(f"\nValidation Ground Truth:")
    print(f"  Known timestomped files: {aggregated_df['ground_truth_label'].sum():,}")
    print(f"  Files with zero nano + ground truth: {((aggregated_df['zero_in_nanoseconds_combined']) & (aggregated_df['ground_truth_label'] == 1)).sum():,}")
    
    # Critical validation check
    known_count = aggregated_df['ground_truth_label'].sum()
    detected_count = ((aggregated_df['zero_in_nanoseconds_combined']) & (aggregated_df['ground_truth_label'] == 1)).sum()
    detection_rate = (detected_count / known_count * 100) if known_count > 0 else 0
    
    print(f"\n  ⚠️  Detection Rate: {detected_count}/{known_count} ({detection_rate:.1f}%)")
    
    if detection_rate < 100:
        print(f"  ⚠️  WARNING: {known_count - detected_count} known files missing zero nanoseconds pattern!")
        
print("\n✓ Data loading and merging complete!")
print(f"✓ Output saved to: {OUTPUT_MERGED_CSV}")



FINAL SUMMARY

Data Reduction:
  LogFile: 27,741 → 1,064 (96.2% reduction)
  UsnJrnl: 248,399 → 15,900 (93.6% reduction)
  Merged: 18,215 records
  Unique files: 3,535 files

Evidence Sources:
  LogFile only: 3
  UsnJrnl only: 2,701
  Both sources (cross-artifact): 831

Key Features:
  Zero nanoseconds: 747
  Cross-artifact validation score = 1.0: 831

✓ Data loading and merging complete!
✓ Output saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/14-W43b/data_merged.csv
